# LeetCode 2039 — The Time When the Network Becomes Idle bằng Python

Link đề: https://leetcode.com/problems/the-time-when-the-network-becomes-idle/ · Độ khó: **Medium** · Chủ đề: Array, Breadth-First Search, Graph

Skeleton LeetCode:

```python
class Solution:
    def networkBecomesIdle(self, edges: List[List[int]], patience: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Tách một đề mô phỏng dài dòng thành hai phần độc lập: **khoảng cách ngắn nhất** từ máy chủ chính (BFS trên đồ thị không trọng số) và **công thức thời gian** cho từng máy chủ dữ liệu.
2. Suy ra chính xác **thời điểm gửi lại cuối cùng**: máy `i` gửi ở các mốc `0, p, 2p, …` **nghiêm ngặt trước** thời điểm nhận phản hồi `2d` ⇒ mốc cuối `p · ⌊(2d − 1)/p⌋`.
3. Cẩn thận với các "off-by-one": gửi đúng lúc phản hồi đến hay không, và mạng **rảnh từ giây kế tiếp** (`+1`).
4. Thấy rằng đáp án **không** chỉ do máy xa nhất hay patience nhỏ nhất quyết định — hàm `p · ⌊(t−1)/p⌋` **không đơn điệu** theo `p`.
5. Viết mô phỏng từng giây làm brute force để kiểm chứng công thức.

## 2. Đề bài & ví dụ

Có `n` máy chủ đánh số `0..n−1`, mảng cạnh `edges` (`[u, v]` là kênh nối trực tiếp, truyền mọi thông điệp trong **1 giây**), và mảng `patience` độ dài `n`. Mạng liên thông.

Máy `0` là **máy chủ chính**, các máy khác là máy dữ liệu. Mỗi máy dữ liệu gửi thông điệp tới máy chính rồi chờ phản hồi. Thông điệp đi theo đường **ngắn nhất**; máy chính xử lý **tức thì** và gửi phản hồi ngược theo đường cũ.

Đầu giây `0`, mọi máy dữ liệu gửi thông điệp. Từ giây `1`, đầu mỗi giây, mỗi máy dữ liệu kiểm tra đã nhận phản hồi (kể cả phản hồi vừa tới) chưa:
- Nếu **chưa**: máy `i` **gửi lại** nếu đã trôi qua `patience[i]` giây kể từ lần gửi gần nhất.
- Nếu **rồi**: không gửi lại nữa.

Mạng **rảnh** khi không còn thông điệp nào đang truyền hoặc đang tới máy nào. Trả về **giây sớm nhất** mà từ đó mạng rảnh.

**Ví dụ 1:** `edges = [[0,1],[1,2]], patience = [0,2,1]` → `8`. Máy 2 cách máy chính 2 cạnh: phản hồi đầu về lúc giây 4; nó đã gửi ở giây 0, 1, 2, 3; thông điệp gửi lúc 3 quay về lúc 7 ⇒ rảnh từ giây 8.

**Ví dụ 2:** `edges = [[0,1],[0,2],[1,2]], patience = [0,10,10]` → `3`.

**Ràng buộc:** `2 ≤ n ≤ 10⁵`; `patience[0] = 0`; `1 ≤ patience[i] ≤ 10⁵` với `i ≥ 1`; `1 ≤ len(edges) ≤ min(10⁵, n(n−1)/2)`; không cạnh lặp, không khuyên; đồ thị liên thông.

In [1]:
import random
import time
import heapq
from collections import deque, defaultdict
from typing import List

examples = [(([[0, 1], [1, 2]], [0, 2, 1]), 8), (([[0, 1], [0, 2], [1, 2]], [0, 10, 10]), 3)]
for (edges, patience), exp in examples:
    print(f"edges={edges}, patience={patience} -> kỳ vọng {exp}")

edges=[[0, 1], [1, 2]], patience=[0, 2, 1] -> kỳ vọng 8
edges=[[0, 1], [0, 2], [1, 2]], patience=[0, 10, 10] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1 BFS cho khoảng cách ngắn nhất trên đồ thị không trọng số

Mỗi cạnh tốn đúng 1 giây ⇒ khoảng cách (số giây) = số cạnh trên đường ngắn nhất. BFS từ đỉnh `0` cho `dist[v]` mọi đỉnh trong `O(n + m)`.

In [2]:
def bfs(n, edges, src=0):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    dist = [-1] * n; dist[src] = 0; q = deque([src])
    while q:
        u = q.popleft()
        for v in g[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1; q.append(v)
    return dist

print(bfs(3, [[0, 1], [1, 2]]), bfs(3, [[0, 1], [0, 2], [1, 2]]))

[0, 1, 2] [0, 1, 1]


### 3.2 Thời gian khứ hồi

Thông điệp từ máy `i` (cách `d` cạnh) tới máy chính sau `d` giây, phản hồi về sau thêm `d` giây ⇒ **khứ hồi** `t = 2d`. Thông điệp gửi lúc `s` có phản hồi về lúc `s + t`.

### 3.3 Đếm bội số nhỏ hơn một ngưỡng

Các mốc `0, p, 2p, …` **nhỏ hơn hẳn** `t` (với `t ≥ 1`) là `k·p` với `0 ≤ k ≤ ⌊(t − 1)/p⌋`. Mốc lớn nhất là `p · ⌊(t − 1)/p⌋`.

In [3]:
for t, p in [(4, 1), (4, 2), (4, 4), (4, 5), (8, 3)]:
    sends = [k * p for k in range(0, t) if k * p < t]
    print(f"t={t}, p={p}: các mốc gửi {sends} | công thức mốc cuối {p * ((t - 1) // p)}")

t=4, p=1: các mốc gửi [0, 1, 2, 3] | công thức mốc cuối 3
t=4, p=2: các mốc gửi [0, 2] | công thức mốc cuối 2
t=4, p=4: các mốc gửi [0] | công thức mốc cuối 0
t=4, p=5: các mốc gửi [0] | công thức mốc cuối 0
t=8, p=3: các mốc gửi [0, 3, 6] | công thức mốc cuối 6


### 3.4 "Rảnh từ giây nào?"

Nếu thông điệp cuối cùng **tới nơi** ở giây `X`, thì trong giây `X` mạng vẫn còn hoạt động; mạng rảnh **từ đầu giây `X + 1`**.

## 4. Ý tưởng nền tảng / trực giác

Các máy dữ liệu **không ảnh hưởng lẫn nhau** (kênh truyền "bao nhiêu thông điệp cũng được"). Vậy chỉ cần tính cho từng máy `i` thời điểm thông điệp cuối cùng của nó quay về, rồi lấy `max`.

Với máy `i`: `t = 2·dist[i]`, `p = patience[i]`.
- Phản hồi đầu tiên về lúc `t`. Ở đầu giây `t`, máy thấy phản hồi ⇒ **không** gửi lại nữa (kể cả nếu `t` là bội của `p`).
- Các lần gửi: mốc `k·p < t` ⇒ mốc cuối `last = p · ⌊(t − 1)/p⌋`.
- Thông điệp gửi lúc `last` về lúc `last + t` ⇒ máy `i` "rảnh" từ `last + t + 1`.

Đáp án `= max_i (p·⌊(t−1)/p⌋ + t + 1)`.

Tính tay **Ví dụ 1**:

| máy | d | t = 2d | p | mốc gửi | last | last + t + 1 |
|---|---|---|---|---|---|---|
| 1 | 1 | 2 | 2 | 0 | 0 | 3 |
| 2 | 2 | 4 | 1 | 0,1,2,3 | 3 | **8** |

In [4]:
(edges, patience), _ = examples[0]
dist = bfs(len(patience), edges)
for i in range(1, len(patience)):
    t, p = 2 * dist[i], patience[i]
    last = p * ((t - 1) // p)
    print(f"máy {i}: d={dist[i]}, t={t}, p={p}, last={last}, rảnh từ {last + t + 1}")

máy 1: d=1, t=2, p=2, last=0, rảnh từ 3
máy 2: d=2, t=4, p=1, last=3, rảnh từ 8


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng giây

Mô phỏng sát đề: mỗi thông điệp mang "số giây còn lại để về tới máy gốc" (`2d` lúc gửi). Đầu mỗi giây `T`: mọi thông điệp tiến 1 giây; thông điệp về tới gốc đánh dấu máy đã nhận phản hồi; máy chưa nhận phản hồi thì kiểm tra để gửi lại. Khi không còn thông điệp nào ⇒ mạng rảnh từ `T + 1`.

In [5]:
def idle_simulation(edges, patience):
    n = len(patience)
    dist = bfs(n, edges)
    replied = [False] * n
    last_sent = [0] * n
    msgs = [[i, 2 * dist[i]] for i in range(1, n)]          # (máy gốc, số giây còn lại)
    T = 0
    while True:
        T += 1
        nxt = []
        for origin, rem in msgs:
            rem -= 1
            if rem == 0:
                replied[origin] = True                       # phản hồi về tới gốc ở giây T
            else:
                nxt.append([origin, rem])
        msgs = nxt
        for i in range(1, n):
            if not replied[i] and T - last_sent[i] == patience[i]:
                msgs.append([i, 2 * dist[i]]); last_sent[i] = T
        if not msgs:
            return T + 1

for (edges, patience), exp in examples:
    got = idle_simulation(edges, patience); print(got, "kỳ vọng", exp); assert got == exp

8 kỳ vọng 8
3 kỳ vọng 3


**Độ phức tạp:** số giây tới `≈ 4·max dist` (tới `4·10⁵`), mỗi giây duyệt `n` máy và mọi thông điệp đang bay (có thể `O(n·t/p)`) ⇒ quá chậm với `n = 10⁵`. Chỉ dùng kiểm chứng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (độc lập):** mỗi máy dữ liệu có "dòng thời gian" riêng chỉ phụ thuộc `(d, p)`. Thời điểm mạng rảnh = `max` các thời điểm rảnh riêng.

**Insight 2 (khoảng cách):** "thông điệp đi tối ưu" ⇒ `d` là khoảng cách BFS; đường đi cụ thể không quan trọng.

**Insight 3 (lần gửi cuối):** gửi ở mốc `k·p` chỉ khi **chưa** nhận phản hồi lúc kiểm tra, tức `k·p < t` ⇒ `last = p·⌊(t − 1)/p⌋`. Phản hồi của lần gửi cuối về lúc `last + t`.

**Insight 4 (+1):** mạng rảnh từ giây sau lần tới nơi cuối cùng.

**Lưu ý:** `last` **không đơn điệu** theo `p` (ví dụ `t = 8`: `p = 3 → 6`, `p = 4 → 4`, `p = 5 → 5`, `p = 7 → 7`), và cũng không chỉ máy xa nhất quyết định ⇒ phải duyệt **mọi** máy.

### 6.1 Khảo sát `last(p)` với `t` cố định

In `last = p·⌊(t−1)/p⌋` cho `t = 12` và mọi `p` từ 1 đến 14 để thấy dạng "răng cưa".

In [6]:
t = 12
row = [(p, p * ((t - 1) // p)) for p in range(1, 15)]
print("p   :", " ".join(f"{p:>3}" for p, _ in row))
print("last:", " ".join(f"{v:>3}" for _, v in row))
print("last + t + 1 lớn nhất ứng với p =", max(row, key=lambda x: x[1])[0], "(không phải p nhỏ nhất!)")

p   :   1   2   3   4   5   6   7   8   9  10  11  12  13  14
last:  11  10   9   8  10   6   7   8   9  10  11   0   0   0
last + t + 1 lớn nhất ứng với p = 1 (không phải p nhỏ nhất!)


## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS + công thức (bản nộp)

In [7]:
def idle_bfs_formula(edges, patience):
    n = len(patience)
    dist = bfs(n, edges)
    ans = 0
    for i in range(1, n):
        t, p = 2 * dist[i], patience[i]
        ans = max(ans, p * ((t - 1) // p) + t + 1)
    return ans

print([idle_bfs_formula(*a) for a, _ in examples])

[8, 3]


**Độ phức tạp Cách 2:** BFS `O(n + m)`, công thức `O(n)` ⇒ `O(n + m)` thời gian và bộ nhớ.

### 7.2 Cách 3 — Công thức đếm số lần gửi lại

Số lần gửi lại (không tính lần đầu) là `r = ⌈t/p⌉ − 1` (số bội dương của `p` nhỏ hơn hẳn `t`). Thời điểm rảnh của máy: `r·p + t + 1`. Hai công thức tương đương vì `⌈t/p⌉ − 1 = ⌊(t−1)/p⌋` với `t ≥ 1`.

In [8]:
def idle_ceil(edges, patience):
    n = len(patience); dist = bfs(n, edges); ans = 0
    for i in range(1, n):
        t, p = 2 * dist[i], patience[i]
        r = -(-t // p) - 1                       # ceil(t/p) - 1
        ans = max(ans, r * p + t + 1)
    return ans

print([idle_ceil(*a) for a, _ in examples])
assert all(-(-t // p) - 1 == (t - 1) // p for t in range(1, 200) for p in range(1, 200))

[8, 3]


### 7.3 Cách 4 — BFS theo tầng, tính đáp án ngay trong BFS

Không cần lưu mảng `dist` riêng: khi một đỉnh được lấy ra ở tầng `d`, tính luôn đóng góp của nó. Đây là bản gọn để nộp.

In [9]:
class Solution:
    def networkBecomesIdle(self, edges: List[List[int]], patience: List[int]) -> int:
        n = len(patience)
        g = [[] for _ in range(n)]
        for u, v in edges:
            g[u].append(v)
            g[v].append(u)
        seen = [False] * n
        seen[0] = True
        frontier = [0]
        d, ans = 0, 0
        while frontier:
            d += 1
            t = 2 * d
            nxt = []
            for u in frontier:
                for v in g[u]:
                    if not seen[v]:
                        seen[v] = True
                        nxt.append(v)
                        p = patience[v]
                        ans = max(ans, p * ((t - 1) // p) + t + 1)
            frontier = nxt
        return ans

print([Solution().networkBecomesIdle(*a) for a, _ in examples])

[8, 3]


**Độ phức tạp Cách 4:** `O(n + m)`.

## 8. TRACE từng bước

Trace mô phỏng (Cách 1) trên **Ví dụ 1**, in trạng thái mỗi giây:

In [10]:
def trace_sim(edges, patience, max_T=20):
    n = len(patience); dist = bfs(n, edges)
    replied, last_sent = [False] * n, [0] * n
    msgs = [[i, 2 * dist[i], 0] for i in range(1, n)]       # (gốc, còn lại, giây gửi)
    print(f"giây 0: gửi {[f'{o}@0' for o, _, _ in msgs]}")
    for T in range(1, max_T):
        nxt, events = [], []
        for o, rem, s in msgs:
            if rem - 1 == 0:
                replied[o] = True; events.append(f"phản hồi {o}@{s} về")
            else:
                nxt.append([o, rem - 1, s])
        msgs = nxt
        for i in range(1, n):
            if not replied[i] and T - last_sent[i] == patience[i]:
                msgs.append([i, 2 * dist[i], T]); last_sent[i] = T; events.append(f"máy {i} gửi lại")
        print(f"giây {T}: {', '.join(events) if events else '-':<40} đang bay: {[f'{o}@{s}' for o, _, s in msgs]}")
        if not msgs:
            print(f"=> rảnh từ giây {T + 1}"); return T + 1

assert trace_sim(*examples[0][0]) == 8

giây 0: gửi ['1@0', '2@0']
giây 1: máy 2 gửi lại                            đang bay: ['1@0', '2@0', '2@1']
giây 2: phản hồi 1@0 về, máy 2 gửi lại           đang bay: ['2@0', '2@1', '2@2']
giây 3: máy 2 gửi lại                            đang bay: ['2@0', '2@1', '2@2', '2@3']
giây 4: phản hồi 2@0 về                          đang bay: ['2@1', '2@2', '2@3']
giây 5: phản hồi 2@1 về                          đang bay: ['2@2', '2@3']
giây 6: phản hồi 2@2 về                          đang bay: ['2@3']
giây 7: phản hồi 2@3 về                          đang bay: []
=> rảnh từ giây 8


| giây | sự kiện chính | thông điệp còn bay |
|---|---|---|
| 0 | máy 1, 2 gửi | `1@0, 2@0` |
| 1 | máy 2 gửi lại | `1@0, 2@0, 2@1` |
| 2 | phản hồi `1@0` về; máy 2 gửi lại | `2@0, 2@1, 2@2` |
| 3 | máy 2 gửi lại (lần cuối) | `2@0, 2@1, 2@2, 2@3` |
| 4 | phản hồi `2@0` về ⇒ máy 2 dừng gửi | `2@1, 2@2, 2@3` |
| 5, 6, 7 | các phản hồi `2@1`, `2@2`, `2@3` lần lượt về | rỗng sau giây 7 |
| 8 | **rảnh** | |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — BFS theo tầng và tính `p·⌊(t−1)/p⌋ + t + 1` ngay khi thăm đỉnh.

In [11]:
class Solution:
    def networkBecomesIdle(self, edges: List[List[int]], patience: List[int]) -> int:
        n = len(patience)
        g = [[] for _ in range(n)]
        for u, v in edges:
            g[u].append(v)
            g[v].append(u)
        seen = [False] * n
        seen[0] = True
        frontier = [0]
        d, ans = 0, 0
        while frontier:
            d += 1
            t = 2 * d
            nxt = []
            for u in frontier:
                for v in g[u]:
                    if not seen[v]:
                        seen[v] = True
                        nxt.append(v)
                        p = patience[v]
                        ans = max(ans, p * ((t - 1) // p) + t + 1)
            frontier = nxt
        return ans

for (edges, patience), exp in examples:
    got = Solution().networkBecomesIdle(edges, patience); print(got, exp); assert got == exp

8 8
3 3


## 10. Vì sao thuật toán đúng

**Độc lập.** Kênh truyền không giới hạn số thông điệp và máy chính xử lý tức thì ⇒ thời gian di chuyển của mỗi thông điệp chỉ phụ thuộc đường đi của nó, không phụ thuộc thông điệp khác. Hành vi gửi lại của máy `i` chỉ phụ thuộc phản hồi **của chính nó**. Vậy tập thời điểm hoạt động của mạng là hợp các tập của từng máy, và thời điểm rảnh là `max`.

**Khứ hồi.** Đi tối ưu ⇒ mỗi chiều tốn `d = dist[i]` giây (BFS đúng vì mọi cạnh 1 giây). Thông điệp gửi lúc `s` có phản hồi về lúc `s + 2d = s + t`.

**Lần gửi cuối.** Phản hồi sớm nhất là của thông điệp gửi lúc 0, về lúc `t`. Máy kiểm tra ở đầu mỗi giây `T ≥ 1`, **sau khi** tính cả phản hồi vừa tới. Với `T < t`: chưa có phản hồi ⇒ gửi lại khi `T` là bội của `p` (vì các lần gửi trước đều ở bội của `p`, khoảng cách `p`). Với `T ≥ t`: đã có phản hồi ⇒ không gửi. Vậy tập lần gửi `= {k·p : k·p < t}`, lần cuối `last = p·⌊(t−1)/p⌋`.

**Rảnh.** Thông điệp cuối của máy `i` về lúc `last + t`; trong giây đó mạng còn hoạt động ⇒ máy `i` hết ảnh hưởng từ `last + t + 1`. Lấy `max`. ∎

### 10.1 Kiểm chứng thực nghiệm trên đồ thị ngẫu nhiên

So mô phỏng từng giây với công thức trên nhiều đồ thị liên thông ngẫu nhiên nhỏ, với `patience` ngẫu nhiên (kể cả `p` lớn hơn khứ hồi và `p` chia hết khứ hồi).

In [12]:
def rand_graph(rng, n, extra):
    edges = []
    for v in range(1, n):
        edges.append([rng.randrange(v), v])                  # cây ngẫu nhiên đảm bảo liên thông
    have = {tuple(sorted(e)) for e in edges}
    cap = n * (n - 1) // 2 - (n - 1)
    for _ in range(min(extra, cap)):
        while True:
            u, v = rng.sample(range(n), 2)
            if tuple(sorted((u, v))) not in have:
                have.add(tuple(sorted((u, v)))); edges.append([u, v]); break
    rng.shuffle(edges)
    return edges

rng = random.Random(1010)
for _ in range(400):
    n = rng.randint(2, 9)
    edges = rand_graph(rng, n, rng.randint(0, 6))
    patience = [0] + [rng.randint(1, 12) for _ in range(n - 1)]
    assert idle_simulation(edges, patience) == idle_bfs_formula(edges, patience) == Solution().networkBecomesIdle(edges, patience)
print("Mô phỏng từng giây = công thức trên 400 đồ thị ngẫu nhiên")

Mô phỏng từng giây = công thức trên 400 đồ thị ngẫu nhiên


## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 10⁵` đỉnh, `m ≤ 10⁵` cạnh.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng giây | `O(T_max · (n + #thông điệp))` | `O(n + #thông điệp)` |
| 2. BFS + công thức | `O(n + m)` | `O(n + m)` |
| 3. BFS + `⌈t/p⌉ − 1` | `O(n + m)` | `O(n + m)` |
| 4. BFS theo tầng | `O(n + m)` | `O(n + m)` |

In [13]:
rng = random.Random(2039)
n = 10 ** 5
edges_big = rand_graph(rng, n, 0)                       # cây ngẫu nhiên 10^5 đỉnh
path = [[i, i + 1] for i in range(n - 1)]               # đường thẳng: khoảng cách lớn nhất
pat = [0] + [rng.randint(1, 10 ** 5) for _ in range(n - 1)]
for name, f in [("bfs_formula", idle_bfs_formula), ("ceil", idle_ceil), ("submit", Solution().networkBecomesIdle)]:
    t = time.perf_counter(); r1 = f(edges_big, pat); r2 = f(path, pat)
    print(f"{name:<12}: cây {r1}, đường {r2} ({(time.perf_counter() - t) * 1000:.0f} ms)")
small = [[i, i + 1] for i in range(100)]; ps = [0] + [1] * 100
t = time.perf_counter(); idle_simulation(small, ps); print(f"mô phỏng đường 101 đỉnh, p = 1: {(time.perf_counter() - t) * 1000:.0f} ms")

bfs_formula : cây 63, đường 399721 (152 ms)
ceil        : cây 63, đường 399721 (112 ms)


submit      : cây 63, đường 399721 (148 ms)


mô phỏng đường 101 đỉnh, p = 1: 215 ms


## 12. Bộ test

In [14]:
ALL = {"bfs_formula": idle_bfs_formula, "ceil": idle_ceil, "submit": Solution().networkBecomesIdle}
def run_all(edges, patience, brute=True):
    res = {k: f(edges, patience) for k, f in ALL.items()}
    if brute:
        res["sim"] = idle_simulation(edges, patience)
    assert len(set(res.values())) == 1, (edges, patience, res)
    return res["submit"]

for (edges, patience), exp in examples:
    assert run_all(edges, patience) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kỳ vọng kiểm bằng mô phỏng trong run_all
edge_cases = [
    (([[0, 1]], [0, 1]), 4),            # d=1, t=2, p=1: gửi 0,1; về 3 -> rảnh 4
    (([[0, 1]], [0, 2]), 3),            # p = t: không gửi lại
    (([[0, 1]], [0, 100]), 3),
    (([[0, 1], [1, 2], [2, 3]], [0, 100, 100, 5]), 12),   # d=3, t=6, p=5: last=5 -> 12
    (([[0, 1], [1, 2], [2, 3]], [0, 100, 100, 3]), 10),   # t=6, p=3: last=3 -> 10
    (([[0, 1], [0, 2]], [0, 1, 1]), 4),
]
for (edges, patience), exp in edge_cases:
    assert run_all(edges, patience) == exp, (edges, patience, run_all(edges, patience))
rng = random.Random(20390)
for _ in range(400):
    n = rng.randint(2, 12)
    edges = rand_graph(rng, n, rng.randint(0, 10))
    run_all(edges, [0] + [rng.randint(1, 20) for _ in range(n - 1)])
for _ in range(5):
    n = rng.randint(2, 5000)
    edges = rand_graph(rng, n, rng.randint(0, 5000))
    run_all(edges, [0] + [rng.randint(1, 10 ** 5) for _ in range(n - 1)], brute=False)
print("Ví dụ, 6 edge case, 400 đồ thị nhỏ (mô phỏng), 5 đồ thị lớn: OK")

Ví dụ, 6 edge case, 400 đồ thị nhỏ (mô phỏng), 5 đồ thị lớn: OK


### 12.1 Kiểm tra tính chất

- Đáp án `≥ 2·max dist + 1` (thông điệp đầu tiên của máy xa nhất).
- Đáp án `≤ 4·max dist` (vì `last ≤ t − 1`).
- Đặt mọi `patience` rất lớn (không máy nào gửi lại) thì đáp án đúng bằng `2·max dist + 1`.

In [15]:
rng = random.Random(121)
f = Solution().networkBecomesIdle
for _ in range(500):
    n = rng.randint(2, 30); edges = rand_graph(rng, n, rng.randint(0, 20))
    pat = [0] + [rng.randint(1, 30) for _ in range(n - 1)]
    D = max(bfs(n, edges)); ans = f(edges, pat)
    assert 2 * D + 1 <= ans <= 4 * D
    assert f(edges, [0] + [10 ** 5] * (n - 1)) == 2 * D + 1
print("Cận 2D + 1 ≤ đáp án ≤ 4D và trường hợp patience lớn: đúng trên 500 đồ thị")

Cận 2D + 1 ≤ đáp án ≤ 4D và trường hợp patience lớn: đúng trên 500 đồ thị


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `⌊t/p⌋` thay vì `⌊(t−1)/p⌋` (gửi lại cả lúc phản hồi vừa tới)

In [16]:
def wrong_floor(edges, patience):
    dist = bfs(len(patience), edges)
    return max(patience[i] * ((2 * dist[i]) // patience[i]) + 2 * dist[i] + 1 for i in range(1, len(patience)))   # SAI

for (edges, patience), exp in examples:
    print("sai :", wrong_floor(edges, patience), "| đúng:", exp)

sai : 9 | đúng: 8
sai : 3 | đúng: 3


**Vì sao sai:** ở đầu giây `t`, máy kiểm tra **cả phản hồi vừa tới** ⇒ đã nhận phản hồi ⇒ không gửi lại dù `t` là bội của `p`. Ví dụ 1: máy 2 có `t = 4`, `p = 1`; công thức sai cho mốc gửi cuối 4 ⇒ 9. **Sửa:** `⌊(t − 1)/p⌋`.

### Lỗi 2 — Quên `+1`

In [17]:
def wrong_no_plus1(edges, patience):
    dist = bfs(len(patience), edges)
    return max(patience[i] * ((2 * dist[i] - 1) // patience[i]) + 2 * dist[i] for i in range(1, len(patience)))   # SAI

for (edges, patience), exp in examples:
    print("sai :", wrong_no_plus1(edges, patience), "| đúng:", exp)

sai : 7 | đúng: 8
sai : 2 | đúng: 3


**Vì sao sai:** thông điệp cuối **tới nơi** ở giây `last + t`, mạng còn hoạt động trong giây đó; đề hỏi giây **bắt đầu rảnh**. **Sửa:** `+ 1`.

### Lỗi 3 — Chỉ xét máy **xa nhất**

In [18]:
def wrong_farthest(edges, patience):
    dist = bfs(len(patience), edges); D = max(dist)
    return max(patience[i] * ((2 * D - 1) // patience[i]) + 2 * D + 1 for i in range(1, len(patience)) if dist[i] == D)   # SAI

# ví dụ tự dựng (đối chiếu lại đề gốc): máy 5 gần hơn nhưng patience nhỏ
chain = [[i, i + 1] for i in range(5)]                  # 0-1-2-3-4-5 : máy 5 cách 5
branch = [[0, 6], [6, 7], [7, 8], [8, 9], [9, 10], [10, 11]]   # máy 11 cách 6
e4 = chain + branch
p4 = [0] * 12
for i in range(1, 12): p4[i] = 100
p4[5] = 1                                               # máy 5: d=5, p=1
print("sai :", wrong_farthest(e4, p4), "| đúng:", Solution().networkBecomesIdle(e4, p4), "| mô phỏng:", idle_simulation(e4, p4))

sai : 13 | đúng: 20 | mô phỏng: 20


**Vì sao sai:** trong đồ thị trên (nhánh `0–1–…–5` và nhánh `0–6–…–11`), máy 5 (`t = 10`, `p = 1`) gửi lại tới mốc 9 ⇒ rảnh từ 20; máy xa nhất 11 (`t = 12`, `p = 100`) chỉ rảnh từ 13. **Sửa:** duyệt **mọi** máy.

### Lỗi 4 — Ở mỗi tầng chỉ dùng `patience` nhỏ nhất

In [19]:
def wrong_min_patience(edges, patience):
    dist = bfs(len(patience), edges); best = {}
    for i in range(1, len(patience)):
        best[dist[i]] = min(best.get(dist[i], 10 ** 9), patience[i])
    return max(p * ((2 * d - 1) // p) + 2 * d + 1 for d, p in best.items())   # SAI

edges5 = [[0, 1], [1, 2], [2, 3], [2, 4]]; patience5 = [0, 100, 100, 3, 5]   # tự dựng (đối chiếu lại đề gốc): máy 3, 4 cùng d = 3, t = 6
print("sai :", wrong_min_patience(edges5, patience5), "| đúng:", Solution().networkBecomesIdle(edges5, patience5), "| mô phỏng:", idle_simulation(edges5, patience5))

sai : 10 | đúng: 12 | mô phỏng: 12


**Vì sao sai:** `last = p·⌊(t−1)/p⌋` **không đơn điệu** theo `p` (mục 6.1). Với `t = 6`: `p = 3` cho `last = 3`, còn `p = 5` cho `last = 5` — patience **lớn hơn** lại gửi muộn hơn. **Sửa:** tính cho từng máy.

### Lỗi 5 — Dùng DFS (không phải khoảng cách ngắn nhất)

In [20]:
def dfs_depth(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    depth = [-1] * n
    def visit(u, d):                                   # DFS đệ quy: đi sâu trước
        depth[u] = d
        for v in g[u]:
            if depth[v] == -1:
                visit(v, d + 1)
    visit(0, 0)
    return depth

def wrong_dfs(edges, patience):
    dist = dfs_depth(len(patience), edges)                                   # SAI: độ sâu DFS
    return max(patience[i] * ((2 * dist[i] - 1) // patience[i]) + 2 * dist[i] + 1 for i in range(1, len(patience)))

edges, patience = [[0, 1], [1, 2], [0, 2]], [0, 5, 5]      # tam giác (tự dựng, đối chiếu lại đề gốc)
print("độ sâu DFS:", dfs_depth(3, [[0, 2], [2, 1], [0, 1]]), "| BFS:", bfs(3, edges))
print("sai :", wrong_dfs([[0, 2], [2, 1], [0, 1]], patience), "| đúng:", Solution().networkBecomesIdle(edges, patience))

độ sâu DFS: [0, 2, 1] | BFS: [0, 1, 1]
sai : 5 | đúng: 3


**Vì sao sai:** DFS có thể đi đường vòng; đề nói thông điệp đi **tối ưu**. Ở đây độ sâu DFS phụ thuộc thứ tự cạnh. **Sửa:** BFS.

### Lỗi 6 — Mô phỏng từng giây với dữ liệu lớn

In [21]:
for L in (50, 100, 150):
    e = [[i, i + 1] for i in range(L)]; p = [0] + [1] * L
    t = time.perf_counter(); idle_simulation(e, p); ts = time.perf_counter() - t
    t = time.perf_counter(); Solution().networkBecomesIdle(e, p); tf = time.perf_counter() - t
    print(f"đường {L + 1} đỉnh, p = 1: mô phỏng {ts * 1000:6.0f} ms | BFS + công thức {tf * 1000:.2f} ms")

đường 51 đỉnh, p = 1: mô phỏng     11 ms | BFS + công thức 0.02 ms


đường 101 đỉnh, p = 1: mô phỏng    217 ms | BFS + công thức 0.03 ms


đường 151 đỉnh, p = 1: mô phỏng    873 ms | BFS + công thức 0.04 ms


**Vì sao không tốt:** với `p = 1`, mỗi máy có `~2d` thông điệp đang bay ⇒ tổng `O(L²)` thông điệp, lặp `O(L)` giây ⇒ `O(L³)`. **Sửa:** công thức.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề mô phỏng dài + `n = 10⁵` ⇒ tìm **công thức đóng** cho từng thực thể, rồi `max`/`sum`.
- Cạnh trọng số 1 + "đi tối ưu" ⇒ **BFS**; trọng số khác ⇒ Dijkstra.
- Đếm các mốc `0, p, 2p, …` **nhỏ hơn hẳn** `t`: `⌊(t − 1)/p⌋ + 1` mốc, mốc cuối `p·⌊(t − 1)/p⌋`.
- Kiểm tra off-by-one bằng ví dụ mà `t` chia hết cho `p` (ở đây Ví dụ 1).
- Viết mô phỏng nhỏ để đối chiếu — đề dạng này rất dễ sai một đơn vị.

## 15. Biến thể của bài

1. **Cạnh có trọng số** (mỗi kênh tốn `w` giây) ⇒ Dijkstra thay BFS, công thức giữ nguyên (Bài tập 5).
2. **Máy chính xử lý mất `c` giây** ⇒ khứ hồi `t = 2d + c`.
3. **743 Network Delay Time**: thời điểm tín hiệu tới đỉnh cuối cùng — chỉ cần `max dist`.
4. **Hỏi số thông điệp tối đa đang bay cùng lúc** — cần quét sự kiện (cộng/trừ tại thời điểm gửi/nhận).
5. **Patience thay đổi theo thời gian** (gửi lại sau `p`, `2p`, `4p`… — exponential backoff) — cộng dồn tới khi `≥ t`.

### 15.1 Code biến thể: exponential backoff, đối chiếu mô phỏng

In [22]:
def idle_backoff(edges, patience):
    # máy i gửi lần đầu ở 0, rồi chờ p, 2p, 4p, ... giữa các lần gửi lại
    dist = bfs(len(patience), edges); ans = 0
    for i in range(1, len(patience)):
        t, gap, last = 2 * dist[i], patience[i], 0
        while last + gap < t:
            last += gap; gap *= 2
        ans = max(ans, last + t + 1)
    return ans

def sim_backoff(edges, patience):
    n = len(patience); dist = bfs(n, edges)
    replied, last_sent, gap = [False] * n, [0] * n, patience[:]
    msgs = [[i, 2 * dist[i]] for i in range(1, n)]; T = 0
    while True:
        T += 1; nxt = []
        for o, r in msgs:
            if r - 1 == 0: replied[o] = True
            else: nxt.append([o, r - 1])
        msgs = nxt
        for i in range(1, n):
            if not replied[i] and T - last_sent[i] == gap[i]:
                msgs.append([i, 2 * dist[i]]); last_sent[i] = T; gap[i] *= 2
        if not msgs:
            return T + 1

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 9); e = rand_graph(rng, n, rng.randint(0, 5)); p = [0] + [rng.randint(1, 6) for _ in range(n - 1)]
    assert idle_backoff(e, p) == sim_backoff(e, p)
print("Backoff: công thức = mô phỏng trên 300 đồ thị;", idle_backoff([[0, 1], [1, 2]], [0, 2, 1]))

Backoff: công thức = mô phỏng trên 300 đồ thị; 8


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng giây | Sát đề | đa thức lớn | lớn | Kiểm chứng |
| 2. BFS + `⌊(t−1)/p⌋` | Công thức từng máy | `O(n+m)` | `O(n+m)` | Dễ đọc |
| 3. BFS + `⌈t/p⌉ − 1` | Đếm lần gửi lại | `O(n+m)` | `O(n+m)` | Cách nhìn khác |
| 4. BFS theo tầng | Tính khi thăm đỉnh | `O(n+m)` | `O(n+m)` | **Bản nộp** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[[0,1]], [0,1]`; (b) `[[0,1]], [0,3]`; (c) `[[0,1],[1,2],[2,3]], [0,9,9,4]`; (d) `[[0,1],[0,2]], [0,2,1]`.

**Bài 2 (tính tay):** Với `t = 10`, tính `last` cho `p = 1..11` và cho biết `p` nào cho `last` lớn nhất.

**Bài 3 (điền TODO):** Hoàn thiện `server_idle(d, p)` trả về giây bắt đầu rảnh của một máy.

**Bài 4 (tự viết lại):** Viết `idle_per_server(edges, patience)` trả danh sách thời điểm rảnh của từng máy dữ liệu (chỉ số 1..n−1).

**Bài 5 (mini challenge):** Mỗi cạnh có trọng số `w` (giây) — `edges[i] = [u, v, w]`. Viết `idle_weighted(edges, patience)` dùng Dijkstra; đối chiếu một mô phỏng từng giây dùng khoảng cách Floyd–Warshall.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def server_idle(d, p):
    # TODO: t = 2 * d; last = p * ((t - 1) // p); return last + t + 1
    pass

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def idle_per_server(edges, patience):
    # TODO: BFS lấy dist, rồi áp server_idle cho từng máy
    pass

def idle_weighted(edges, patience):
    # TODO: Dijkstra từ 0 trên đồ thị có trọng số; công thức như cũ với t = 2 * dist[i]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: BFS. Mức 2: `t = 2d`, `last = p·⌊(t−1)/p⌋`. Mức 3: (a) `t = 2, p = 1` ⇒ `last = 1` ⇒ `4`; (b) `p = 3 > t` ⇒ `last = 0` ⇒ `3`; (c) máy 3: `t = 6, p = 4` ⇒ `last = 4` ⇒ `11` (máy 1, 2 với `p = 9` cho `3`, `5`); (d) máy 1: `p = 2 = t` ⇒ `3`, máy 2: `p = 1` ⇒ `4` ⇒ `4`.

**Bài 2.** Mức 1: `⌊9/p⌋·p`. Mức 2: `p = 1..11` cho `9, 8, 9, 8, 5, 6, 7, 8, 9, 0, 0`. Mức 3: lớn nhất `9` ở `p ∈ {1, 3, 9}`.

**Bài 3.** Mức 1: khứ hồi. Mức 2: lần gửi cuối. Mức 3: `+ t + 1`.

**Bài 4.** Mức 1: dùng lại `bfs`. Mức 2: list comprehension. Mức 3: `max` của danh sách = đáp án bài gốc.

**Bài 5.** Mức 1: `heapq` với `(dist, node)`. Mức 2: bỏ qua phần tử cũ trong heap. Mức 3: mô phỏng chỉ cần thay `2·dist` BFS bằng `2·dist` có trọng số.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2, 3
f = Solution().networkBecomesIdle
b1 = [f(e, p) for e, p in [([[0, 1]], [0, 1]), ([[0, 1]], [0, 3]), ([[0, 1], [1, 2], [2, 3]], [0, 9, 9, 4]), ([[0, 1], [0, 2]], [0, 2, 1])]]
print("Bài 1:", b1); assert b1 == [4, 3, 11, 4]
t = 10; b2 = [p * ((t - 1) // p) for p in range(1, 12)]
print("Bài 2:", b2); assert b2 == [9, 8, 9, 8, 5, 6, 7, 8, 9, 0, 0]
def server_idle(d, p):
    t = 2 * d
    return p * ((t - 1) // p) + t + 1
assert server_idle(2, 1) == 8 and server_idle(1, 2) == 3
print("Bài 3: OK")

Bài 1: [4, 3, 11, 4]
Bài 2: [9, 8, 9, 8, 5, 6, 7, 8, 9, 0, 0]
Bài 3: OK


In [26]:
# Bài 4, 5
def idle_per_server(edges, patience):
    dist = bfs(len(patience), edges)
    return [server_idle(dist[i], patience[i]) for i in range(1, len(patience))]

def dijkstra(n, wedges, src=0):
    g = [[] for _ in range(n)]
    for u, v, w in wedges:
        g[u].append((v, w)); g[v].append((u, w))
    dist = [float("inf")] * n; dist[src] = 0; pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w; heapq.heappush(pq, (dist[v], v))
    return dist

def idle_weighted(edges, patience):
    dist = dijkstra(len(patience), edges)
    return max(server_idle(dist[i], patience[i]) for i in range(1, len(patience)))

def idle_weighted_sim(edges, patience):
    n = len(patience); INF = float("inf")
    D = [[INF] * n for _ in range(n)]
    for i in range(n): D[i][i] = 0
    for u, v, w in edges: D[u][v] = D[v][u] = min(D[u][v], w)
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if D[i][k] + D[k][j] < D[i][j]: D[i][j] = D[i][k] + D[k][j]
    replied, last_sent = [False] * n, [0] * n
    msgs = [[i, 2 * D[i][0]] for i in range(1, n)]; T = 0
    while True:
        T += 1; nxt = []
        for o, r in msgs:
            if r - 1 == 0: replied[o] = True
            else: nxt.append([o, r - 1])
        msgs = nxt
        for i in range(1, n):
            if not replied[i] and T - last_sent[i] == patience[i]:
                msgs.append([i, 2 * D[i][0]]); last_sent[i] = T
        if not msgs: return T + 1

assert max(idle_per_server(*examples[0][0])) == 8
rng = random.Random(5)
for _ in range(200):
    n = rng.randint(2, 7); e = [x + [rng.randint(1, 4)] for x in rand_graph(rng, n, rng.randint(0, 4))]
    p = [0] + [rng.randint(1, 10) for _ in range(n - 1)]
    assert idle_weighted(e, p) == idle_weighted_sim(e, p)
print("Bài 4, 5: OK |", idle_per_server(*examples[0][0]), idle_weighted([[0, 1, 3], [1, 2, 1], [0, 2, 10]], [0, 2, 5]))

Bài 4, 5: OK | [3, 8] 14


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
dist = BFS từ 0
ans = 0
for i in 1..n-1:
    t, p = 2 * dist[i], patience[i]
    last = p * ((t - 1) // p)          # lần gửi cuối (nghiêm ngặt trước t)
    ans = max(ans, last + t + 1)       # phản hồi cuối về ở last + t, rảnh từ +1
return ans
```

**Checklist:**
- [ ] Khoảng cách bằng **BFS** (không phải DFS)?
- [ ] Khứ hồi `t = 2d`?
- [ ] `⌊(t − 1)/p⌋` — không gửi lại đúng lúc phản hồi đến?
- [ ] `+1` cho "rảnh từ giây kế tiếp"?
- [ ] Duyệt **mọi** máy (không chỉ máy xa nhất / patience nhỏ nhất)?